# 📚 PyTorch Practice Notebook - Lecture 4:Transfer Learning Mastery Exercises

## Progressive Transfer Learning on CIFAR-10

---

## 🎯 Objective of This Practice

This notebook is a **full end-to-end deep learning practice** designed to build **both intuition and engineering discipline** around:

- Training CNNs **from scratch**
- Understanding the **limits of training from scratch**
- Applying **transfer learning strategies**
- Evaluating **progressive fine-tuning**

By the end of this notebook, you will have:
- Built and evaluated multiple CNN pipelines
- Identified when transfer learning becomes necessary
- Implemented **professional-grade progressive fine-tuning**
- Documented all experiments reproducibly

---

## 🔍 Research Question

**RQ1:**  
Does **progressive fine-tuning**—where a single model is trained by gradually unfreezing layers—outperform **separate transfer learning strategies**, in which head-only, partial, and full fine-tuning models are trained independently from ImageNet initialization?

---

## 🧠 Motivation

Standard transfer learning comparisons often rely on training **separate models** for each fine-tuning strategy. While this simplifies analysis, it introduces an inefficiency:

> **Previously learned task-specific representations are discarded** when moving from one strategy to another.

This notebook evaluates whether a **progressive fine-tuning pipeline**, which preserves and refines learned weights across phases, leads to:
- Higher accuracy
- Faster convergence
- More stable training

---

## 🧪 Hypotheses

We hypothesize that **progressive transfer learning** will:

- **H1:** Achieve higher final validation accuracy  
- **H2:** Converge faster in later training stages  
- **H3:** Exhibit smoother and more stable learning curves  
- **H4:** Reduce catastrophic forgetting during deeper fine-tuning  



In [ ]:
import torch
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, Subset
import matplotlib.pyplot as plt
import numpy as np
import torch.nn as nn
import torch.optim as optim
import time
from torchvision import datasets, transforms, models
from tqdm import tqdm
import torchvision
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report


seed = 42
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

print(f"GPU available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

---

# PART 1 — Dataset Preparation (CIFAR-10)

## 📦 Task 1: Load and Inspect CIFAR-10

### Your Responsibilities
- Load the CIFAR-10 dataset
- Inspect image shapes and class labels
- Visualize sample images per class

---




In [ ]:
# ==========================================
# TASK 1: LOAD AND INSPECT CIFAR-10
# ==========================================


# ==========================================
# 1. Load Dataset (No normalization for visualization)
# ==========================================
vis_transform = transforms.ToTensor()  # Converts PIL image to [0,1] float tensor

vis_dataset = torchvision.datasets.CIFAR10(
    root='./data',
    train=True,
    download=True,
    transform=vis_transform
)

classes = vis_dataset.classes  # ['airplane', 'automobile', ..., 'truck']

# ==========================================
# 2. Inspect Image Shapes and Class Labels
# ==========================================
print("=" * 50)
print("  CIFAR-10 Dataset Inspection")
print("=" * 50)
print(f"Total training samples : {len(vis_dataset)}")
print(f"Number of classes      : {len(classes)}")
print(f"Class labels           : {classes}")

# Grab a single sample to inspect tensor shape
sample_img, sample_label = vis_dataset[0]
print(f"\nImage tensor shape     : {sample_img.shape}  → (C, H, W)")
print(f"Pixel value range      : [{sample_img.min():.2f}, {sample_img.max():.2f}]")
print(f"Sample label           : {sample_label} → '{classes[sample_label]}'")
print("=" * 50)

# ==========================================
# 3. Visualize Sample Images Per Class
# ==========================================
SAMPLES_PER_CLASS = 5
num_classes = len(classes)

fig, axes = plt.subplots(
    num_classes, SAMPLES_PER_CLASS,
    figsize=(SAMPLES_PER_CLASS * 2, num_classes * 2)
)
fig.suptitle("CIFAR-10 — Sample Images per Class", fontsize=14, fontweight='bold', y=1.01)

# Track how many samples we've collected per class
class_counts = {i: 0 for i in range(num_classes)}

for img, label in vis_dataset:
    # Skip if we already have enough samples for this class
    if class_counts[label] >= SAMPLES_PER_CLASS:
        continue

    # Convert tensor (C, H, W) → numpy (H, W, C) for matplotlib
    img_np = np.transpose(img.numpy(), (1, 2, 0))  # Shape: (32, 32, 3)

    ax = axes[label, class_counts[label]]
    ax.imshow(img_np)
    ax.axis('off')

    # Add class name label only on the first column
    if class_counts[label] == 0:
        ax.set_ylabel(
            classes[label],
            fontsize=11,
            fontweight='bold',
            rotation=0,
            labelpad=50,
            va='center'
        )

    class_counts[label] += 1

    # Stop early once all classes have enough samples
    if all(count == SAMPLES_PER_CLASS for count in class_counts.values()):
        break

plt.tight_layout()
plt.show()

## 🔧 Task 2: Preprocessing & DataLoaders

### Requirements
- Apply appropriate transforms:
  - Normalization
  - Data augmentation (for training set)
- Split data into:
  - Training
  - Validation
  - Test
- Build PyTorch `DataLoader`s

📌 **Deliverable:**  
Explain *why* each preprocessing step is used.

---

## 🧠 Task 3: Design a CNN Architecture (From Scratch)
### Constraints
- No pretrained weights
- Architecture must include:
  - Convolution blocks
  - Non-linearities
  - Pooling
  - Regularization (Dropout / BatchNorm)

📌 **Goal:**  
Iteratively improve the architecture until performance **plateaus**.

---

In [ ]:
# ==========================================
# TASK 2: PREPROCESSING & DATALOADERS (v2.0)
# ==========================================

import os
import torch
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, Subset

# ==========================================
# 1. Define Transforms
# ==========================================

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),  # upsample from 32x32
    transforms.RandomHorizontalFlip(),
    transforms.RandomCrop(224, padding=4),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],  # ImageNet stats
                         std=[0.229, 0.224, 0.225])
])

test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

# ==========================================
# 2. Load & Split Dataset
# ==========================================

# Load the full training set TWICE to isolate augmentations
train_full = torchvision.datasets.CIFAR10(root='./data', train=True,
                                           download=True, transform=train_transform)
val_full   = torchvision.datasets.CIFAR10(root='./data', train=True,
                                           download=True, transform=test_transform)
test_set   = torchvision.datasets.CIFAR10(root='./data', train=False,
                                           download=True, transform=test_transform)

VAL_SIZE   = 10_000
TRAIN_SIZE = len(train_full) - VAL_SIZE  # 40,000

# Deterministic split
torch.manual_seed(42)
indices       = torch.randperm(len(train_full)).tolist()
train_indices = indices[:TRAIN_SIZE]
val_indices   = indices[TRAIN_SIZE:]

# Create subsets
train_subset = Subset(train_full, train_indices)  # Augmentation ON
val_subset   = Subset(val_full,   val_indices)    # Augmentation OFF

print("=" * 50)
print("  Dataset Split & Loader Configuration")
print("=" * 50)
print(f"Train samples      : {len(train_subset):,}")
print(f"Validation samples : {len(val_subset):,}")
print(f"Test samples       : {len(test_set):,}")

# ==========================================
# 3. DataLoaders (Production Optimized)
# ==========================================
BATCH_SIZE = 128

# Hardware-aware worker allocation (avoids bottlenecking or CPU thrashing)
num_workers = min(4, os.cpu_count() or 1)
print(f"Allocated workers  : {num_workers}")
print("=" * 50)

# Seeded generator for strict worker reproducibility
loader_generator = torch.Generator().manual_seed(42)

train_loader = DataLoader(
    train_subset, 
    batch_size=BATCH_SIZE,
    shuffle=True,  
    num_workers=num_workers, 
    pin_memory=True,
    drop_last=True,                # Drops the incomplete final batch to stabilize BatchNorm
    generator=loader_generator     # Ensures reproducible augmentations across workers
)

val_loader = DataLoader(
    val_subset,   
    batch_size=BATCH_SIZE,
    shuffle=False, 
    num_workers=num_workers, 
    pin_memory=True
)

test_loader = DataLoader(
    test_set,     
    batch_size=BATCH_SIZE,
    shuffle=False, 
    num_workers=num_workers, 
    pin_memory=True
)

# ==========================================
# 4. CNN Architecture
#    Input:  (B, 3, 32, 32)
#    Output: (B, 10)
# ==========================================

class AlexNet(nn.Module):
    def __init__(self, classes=10, dropout_p=0.5):
        super().__init__()
        
        self.feature_extractor = nn.Sequential(
            # 32x32 → 32x32
            nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),  # 32 → 16
            nn.BatchNorm2d(64),

            # 16x16 → 16x16
            nn.Conv2d(64, 192, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),  # 16 → 8
            nn.BatchNorm2d(192),

            # 8x8 → 8x8
            nn.Conv2d(192, 384, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.BatchNorm2d(384),

            nn.Conv2d(384, 256, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.BatchNorm2d(256),

            nn.Conv2d(256, 256, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),  # 8 → 4
            nn.BatchNorm2d(256),
        )

        self.head = nn.Sequential(
            nn.Dropout(dropout_p),
            nn.Linear(256 * 4 * 4, 4096),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout_p),
            nn.Linear(4096, 4096),
            nn.ReLU(inplace=True),
            nn.Linear(4096, classes),
        )

    def forward(self, x):
        x = self.feature_extractor(x)
        x = torch.flatten(x, 1)
        x = self.head(x)
        return x


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = AlexNet(classes=10).to(device)

x = torch.randn(4, 3, 32, 32).to(device)
y = model(x)

print("Input shape :", x.shape)
print("Output shape:", y.shape)

print(model)



## 🧪 Task 4: Scratch Training Experiments

### Experiments to Run
- Vary depth and width
- Try different optimizers
- Adjust learning rates and schedulers

### Record
- Best validation accuracy
- Training stability
- Signs of underfitting / overfitting

---

In [ ]:
def run_experiment(model, train_loader, val_loader, optimizer, criterion, scheduler=None, epochs=20, device="cuda"):
    """
    Runs a single training experiment with AMP (Automatic Mixed Precision).
    """
    model = model.to(device)
    
    # Modern PyTorch: Automatic Mixed Precision (AMP) Scaler
    scaler = torch.amp.GradScaler(device) 
    
    train_accs, val_accs = [], []
    best_val_acc = 0.0

    print("=" * 60)
    # ⬇️ THE FIX IS ON THIS LINE ⬇️
    print(f"EXPERIMENT: Opt={optimizer.__class__.__name__} | LR={optimizer.param_groups[0]['lr']}")   
    print("=" * 60)

    for epoch in range(epochs):
        # -------- Train --------
        model.train()
        correct, total = 0, 0
        
        pbar = tqdm(train_loader, desc=f"Epoch {epoch+1:02d}/{epochs} [Train]")
        for images, labels in pbar:
            images, labels = images.to(device), labels.to(device)

            # SOTA memory optimization
            optimizer.zero_grad(set_to_none=True)

            # Train in Mixed Precision for 2x speed / 0.5x VRAM
            with torch.autocast(device_type=device, dtype=torch.float16):
                outputs = model(images)
                loss = criterion(outputs, labels)

            # Scale gradients to prevent FP16 underflow
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            _, preds = outputs.max(1)
            total += labels.size(0)
            correct += preds.eq(labels).sum().item()

            pbar.set_postfix(Loss=f"{loss.item():.3f}", Acc=f"{100.0 * correct / total:.2f}%")

        train_acc = 100.0 * correct / total
        train_accs.append(train_acc)

        # -------- Validation (NOT TEST) --------
        model.eval()
        correct, total = 0, 0

        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                
                _, preds = outputs.max(1)
                total += labels.size(0)
                correct += preds.eq(labels).sum().item()

        val_acc = 100.0 * correct / total
        val_accs.append(val_acc)
        
        # Track best model
        if val_acc > best_val_acc:
            best_val_acc = val_acc

        # Step scheduler using Validation data
        if scheduler:
            if isinstance(scheduler, torch.optim.lr_scheduler.ReduceLROnPlateau):
                scheduler.step(val_acc)
            else:
                scheduler.step()

        # Diagnostics
        gap = train_acc - val_acc
        warning = ""
        if gap > 15:
            warning = " ⚠️ Overfitting"
        elif train_acc < 50 and val_acc < 50:
            warning = " ⚠️ Underfitting"

        print(f"Epoch {epoch+1:02d} | Train: {train_acc:5.2f}% | Val: {val_acc:5.2f}% | Gap: {gap:5.2f}% {warning}")

    print(f"🏆 Best Validation Accuracy: {best_val_acc:.2f}%")
    return train_accs, val_accs            


In [ ]:

# ==========================================
# RUNNING THE EXPERIMENTS
# ==========================================

# Experiment 1: Adam
model_1 = AlexNet(classes=10)
opt_1 = torch.optim.Adam(model_1.parameters(), lr=1e-3)
run_experiment(model_1, train_loader, val_loader, opt_1, nn.CrossEntropyLoss(), epochs=15)

# Experiment 2: SGD with Momentum (Often beats Adam on CIFAR-10)
model_2 = AlexNet(classes=10)
opt_2 = torch.optim.SGD(model_2.parameters(), lr=0.1, momentum=0.9, weight_decay=5e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(opt_2, T_max=15)
run_experiment(model_2, train_loader, val_loader, opt_2, nn.CrossEntropyLoss(), scheduler=scheduler, epochs=15)


## 🧠 Reflection: Scratch Training Threshold

Answer the following:

- What accuracy ceiling do you hit?
- What limits further improvement?
- Why does CIFAR-10 expose these limits?

📌 **Conclusion:**  
Define the **threshold** where training from scratch becomes inefficient.

---
1. What accuracy ceiling do you hit?

        From your experiments:

        Adam (LR=0.001):
        → Best Validation Accuracy ≈ 86.17%
        SGD (LR=0.1):
        → Best Validation Accuracy ≈ 88.49%

2. What limits further improvement?

    A.Generalization Gap Increasing
    Final gap:
    Adam → ~2.85%
    SGD → ~4.02%
        👉 Model starts to:
                Memorize training data
                Not improve real-world performance

    B.Model Capacity Limitation

        Your CNN (likely a small custom one):

        Cannot capture high-level hierarchical features
        Lacks:
        Deep layers
        Residual connections (like ResNet)
        Advanced feature reuse

        👉 Result: Underfitting complex patterns at higher levels

C.Computation Cost
Computation becomes inefficient when training continues to consume the same time per epoch while validation accuracy improves only marginally.


3. Why does CIFAR-10 expose these limits?
        A. Low Resolution Bottleneck
        32×32 images lose critical spatial details
        → Hard to distinguish:
        cat vs dog
        truck vs automobile

        B. High Intra-class Variation
        Same class looks very different:
        Different angles
        Backgrounds
        Lighting

        C.Inter-class Similarity
        Different classes look similar:
        Cat vs Dog
        Plane vs Bird
        👉 Requires strong feature extraction, not shallow patterns

        D. Limited Data (for deep learning)
        50k samples is not enough for deep models from scratch

# PART 3 — Transfer Learning with Pretrained Models

## 🔁 Task 5: Select a Pretrained Model

Choose **one non-ResNet architecture** from `torchvision`, such as:
- VGG
- DenseNet
- MobileNet
- EfficientNet

Explain why you chose it.

---


### ✅ Chosen Model: MobileNetV2 🎯 Why MobileNetV2?
    1. Efficiency vs Performance Trade-off
    MobileNetV2 is designed to be:
    Lightweight 🪶
    Fast ⚡
    Still highly accurate for Small Image Resolution (32×32)
    like CIFAR-10 images :
    Very low resolution
    Contain limited spatial detail

    2. Pretrained on Large Dataset (ImageNet)
    Learned from 1M+ images
    Already understands:
    Edges
    Textures
    Shapes

    3. Depthwise Separable Convolutions
    Instead of standard convolutions, it uses:
    Depthwise + Pointwise convolutions
    👉 This reduces:
    Parameters
    Computation


    4. Works Well with Small Images (like CIFAR-10)


## ⚙️ Task 6: Apply Transfer Learning Strategies

Implement and evaluate the following **independently**:

### Strategy A — Head-Only Fine-Tuning
- Freeze entire backbone
- Train classifier only

### Strategy B — Partial Fine-Tuning
- Unfreeze last few convolutional blocks
- Train classifier + selected layers

### Strategy C — Full Fine-Tuning
- Unfreeze entire network
- Fine-tune end-to-end

📌 Track:
- Accuracy
- Convergence speed
- Training stability

---

In [ ]:
print("="*60)
print("IMPLEMENTING THREE TRANSFER LEARNING STRATEGIES")
print("="*60)

class TransferLearningModel:
    """Flexible class to implement all three transfer learning strategies"""

    def __init__(self, model_name='mobilenet_v2', num_classes=10, strategy='head_only'):
        """
        Initialize transfer learning model

        Args:
            model_name: Pre-trained model to use
            num_classes: Number of output classes
            strategy: 'head_only', 'partial', or 'full'
        """
        self.model_name = model_name
        self.num_classes = num_classes
        self.strategy = strategy

        print(f"\n🔧 Creating {model_name} with {strategy} strategy...")

        # 1. Load pre-trained model
        self.model = self._load_pretrained_model()

        # 2. Replace the final layer for our task
        self._replace_classifier()

        # 3. Apply the chosen strategy
        self._apply_strategy()

        # 4. Print statistics
        self._print_statistics()

    def _load_pretrained_model(self):
        """Load pre-trained model from torchvision"""
        model_func = getattr(models, self.model_name)
        model = model_func(pretrained=True)
        print(f"  ✅ Loaded {self.model_name} pre-trained on ImageNet")
        return model

    def _replace_classifier(self):
        """Replace the final classification layer"""
        if hasattr(self.model, 'fc'):  # ResNet
            num_features = self.model.fc.in_features
            self.model.fc = nn.Linear(num_features, self.num_classes)
            print(f"  🔄 Replaced final layer: {num_features} → {self.num_classes} features")
        elif hasattr(self.model, 'classifier'):  # Other architectures
            if isinstance(self.model.classifier, nn.Sequential):
                num_features = self.model.classifier[-1].in_features
                self.model.classifier[-1] = nn.Linear(num_features, self.num_classes)
                print(f"  🔄 Replaced final layer: {num_features} → {self.num_classes} features")

    def _apply_strategy(self):
        """Apply the chosen transfer learning strategy"""
        if self.strategy == 'head_only':
            self._freeze_all_except_head()
        elif self.strategy == 'partial':
            self._freeze_early_layers()
        elif self.strategy == 'full':
            self._unfreeze_all()
        else:
            raise ValueError(f"Unknown strategy: {self.strategy}")

    def _freeze_all_except_head(self):
        """Freeze all layers except the final classifier"""
        for name, param in self.model.named_parameters():
            # Only train parameters in the final layer
            if 'fc' not in name and 'classifier' not in name:
                param.requires_grad = False
        print("  🔒 Strategy: Head Only - Only final layer trainable")
            
    def _freeze_early_layers(self):
        # MobileNetV2: features[0..18] + classifier
        # Freeze first ~half of the feature extractor
        layers_to_freeze = [f'features.{i}' for i in range(10)]  # 0–9 frozen, 10–18 trainable

        for name, param in self.model.named_parameters():
            if any(name.startswith(layer) for layer in layers_to_freeze):
                param.requires_grad = False
            else:
                param.requires_grad = True
        print("  ⚡ Strategy: Partial - features[0-9] frozen, features[10-18]+classifier trainable")

    def _unfreeze_all(self):
        """Make all parameters trainable"""
        for param in self.model.parameters():
            param.requires_grad = True
        print("  🔥 Strategy: Full - All layers trainable")

    def _print_statistics(self):
        """Print model statistics"""
        total_params = sum(p.numel() for p in self.model.parameters())
        trainable_params = sum(p.numel() for p in self.model.parameters()
                              if p.requires_grad)
        frozen_params = total_params - trainable_params

        print(f"  📊 Parameters: {total_params:,} total, {trainable_params:,} trainable "
              f"({trainable_params/total_params:.1%})")

        # Count trainable vs frozen layers
        trainable_layers = []
        frozen_layers = []

        for name, param in self.model.named_parameters():
            layer_name = name.split('.')[0]
            if param.requires_grad:
                if layer_name not in trainable_layers:
                    trainable_layers.append(layer_name)
            else:
                if layer_name not in frozen_layers:
                    frozen_layers.append(layer_name)

        print(f"  🏗️  Architecture: {len(trainable_layers)} trainable layers, "
              f"{len(frozen_layers)} frozen layers")

    def get_model(self):
        """Return the model"""
        return self.model

    def get_optimizer(self, lr=0.001):
        """Get appropriate optimizer for the strategy"""
        # Collect trainable parameters
        params_to_optimize = [
            param for param in self.model.parameters()
            if param.requires_grad
        ]

        # Different learning rates for different strategies
        if self.strategy == 'head_only':
            lr = 0.0001  # Smaller LR for head-only
        elif self.strategy == 'partial':
            lr = 0.0005  # Medium LR
        else:  # full
            lr = 0.001  # Standard LR

        optimizer = optim.Adam(params_to_optimize, lr=lr)
        print(f"  ⚙️  Optimizer: Adam with lr={lr}")
        return optimizer

# Test all three strategies
print("\n🧪 TESTING ALL THREE STRATEGIES:")
print("-" * 60)

strategies = ['head_only', 'partial', 'full']
strategy_names = ['Head Only Tuning', 'Partial Fine-Tuning', 'Full Fine-Tuning']

models_dict = {}
for strategy, name in zip(strategies, strategy_names):
    print(f"\n{name}:")
    print("-" * 40)

    # Create model
    tl_model = TransferLearningModel(
        model_name='mobilenet_v2',
        num_classes=10,
        strategy=strategy
    )

    # Store model
    models_dict[strategy] = {
        'model': tl_model.get_model(),
        'optimizer': tl_model.get_optimizer(),
        'name': name
    }

In [ ]:
print("="*60)
print("UNIFIED TRAINING FRAMEWORK")
print("="*60)

class UnifiedTransferLearningTrainer:
    """Train and compare all transfer learning strategies"""

    def __init__(self, device='cuda'):
        self.device = device
        self.results = {}
        self.models = {}

    def train_strategy(self, strategy_name, model, train_loader, val_loader,
                      epochs=5, lr=0.001):
        """Train a single strategy"""
        print(f"\n🚀 Training Strategy: {strategy_name}")
        print("="*50)

        # Move model to device
        model = model.to(self.device)

        # Setup loss and optimizer
        criterion = nn.CrossEntropyLoss()
        optimizer = optim.Adam(
            [p for p in model.parameters() if p.requires_grad],
            lr=lr
        )

        # Learning rate scheduler
        scheduler = optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, mode='max', patience=2, factor=0.5
        )

        # Training history
        history = {
            'epoch': [],
            'train_loss': [], 'train_acc': [],
            'val_loss': [], 'val_acc': [],
            'learning_rate': []
        }

        best_val_acc = 0.0
        best_model_state = None

        for epoch in range(epochs):
            # ----- Training Phase -----
            model.train()
            train_loss = 0.0
            train_correct = 0
            train_total = 0

            train_pbar = tqdm(train_loader, desc=f'Epoch {epoch+1}/{epochs} [Train]')
            for images, labels in train_pbar:
                images, labels = images.to(self.device), labels.to(self.device)

                optimizer.zero_grad()
                outputs = model(images)
                loss = criterion(outputs, labels)
                loss.backward()
                optimizer.step()

                train_loss += loss.item()
                _, predicted = outputs.max(1)
                train_total += labels.size(0)
                train_correct += predicted.eq(labels).sum().item()

                # Update progress bar
                train_pbar.set_postfix({
                    'Loss': f'{loss.item():.3f}',
                    'Acc': f'{100.*train_correct/train_total:.1f}%'
                })

            avg_train_loss = train_loss / len(train_loader)
            train_acc = 100. * train_correct / train_total

            # ----- Validation Phase -----
            model.eval()
            val_loss = 0.0
            val_correct = 0
            val_total = 0

            with torch.no_grad():
                val_pbar = tqdm(val_loader, desc=f'Epoch {epoch+1}/{epochs} [Val]')
                for images, labels in val_pbar:
                    images, labels = images.to(self.device), labels.to(self.device)
                    outputs = model(images)
                    loss = criterion(outputs, labels)

                    val_loss += loss.item()
                    _, predicted = outputs.max(1)
                    val_total += labels.size(0)
                    val_correct += predicted.eq(labels).sum().item()

                    val_pbar.set_postfix({
                        'Acc': f'{100.*val_correct/val_total:.1f}%'
                    })

            avg_val_loss = val_loss / len(val_loader)
            val_acc = 100. * val_correct / val_total

            # Update learning rate
            old_lr = optimizer.param_groups[0]['lr']
            scheduler.step(val_acc)
            new_lr = optimizer.param_groups[0]['lr']

            if new_lr < old_lr:
                print(f"    📉 Learning rate reduced: {old_lr:.6f} → {new_lr:.6f}")

            # Save history
            history['epoch'].append(epoch + 1)
            history['train_loss'].append(avg_train_loss)
            history['train_acc'].append(train_acc)
            history['val_loss'].append(avg_val_loss)
            history['val_acc'].append(val_acc)
            history['learning_rate'].append(new_lr)

            # Print epoch summary
            print(f"\n  📈 Epoch {epoch+1} Summary:")
            print(f"    Train Loss: {avg_train_loss:.4f}, Train Acc: {train_acc:.2f}%")
            print(f"    Val Loss: {avg_val_loss:.4f}, Val Acc: {val_acc:.2f}%")
            print(f"    Learning Rate: {new_lr:.6f}")

            # Save best model
            if val_acc > best_val_acc:
                best_val_acc = val_acc
                best_model_state = model.state_dict().copy()
                print(f"    🏆 New best model! Val Acc: {best_val_acc:.2f}%")

        # Store results
        self.results[strategy_name] = {
            'history': history,
            'best_val_acc': best_val_acc,
            'best_model_state': best_model_state
        }
        self.models[strategy_name] = model

        print(f"\n✅ {strategy_name} training complete!")
        print(f"   Best validation accuracy: {best_val_acc:.2f}%")

        return history, best_model_state

    def compare_strategies(self):
        """Compare all trained strategies"""
        if not self.results:
            print("No results to compare!")
            return

        print("\n" + "="*60)
print("COMPARING ALL TRANSFER LEARNING STRATEGIES")
print("="*60)

# Create trainer
trainer = UnifiedTransferLearningTrainer(device=device)

# Define training parameters for each strategy
training_configs = {
    'head_only': {
        'epochs': 5,
        'lr': 0.0001,
        'description': 'Head Only Tuning (Fastest)'
    },
    'partial': {
        'epochs': 8,
        'lr': 0.0005,
        'description': 'Partial Fine-Tuning (Balanced)'
    },
    'full': {
        'epochs': 10,
        'lr': 0.001,
        'description': 'Full Fine-Tuning (Most Accurate)'
    }
}

# Train all strategies
for strategy, config in training_configs.items():
    # Create model with this strategy
    tl_model = TransferLearningModel(
        model_name='mobilenet_v2',
        num_classes=10,
        strategy=strategy
    )

    model = tl_model.get_model()

    print(f"\n🎯 Training: {config['description']}")
    print(f"   Epochs: {config['epochs']}, Learning Rate: {config['lr']}")

    # Train the model
    history, best_state = trainer.train_strategy(
        strategy_name=strategy,
        model=model,
        train_loader=train_loader,
        val_loader=test_loader,  # Using test as validation for demonstration
        epochs=config['epochs'],
        lr=config['lr']
    )

print("\n✅ All strategies trained successfully!")

# Compare strategies
trainer.compare_strategies()

# PART 4 — Progressive Transfer Learning (Main Experiment)

## 🧪 Experimental Design

### Progressive Fine-Tuning Pipeline

| Phase | Trainable Parameters | Objective | 
|------|---------------------|-----------| 
| Phase 1 | Classification head only | Learn task-specific boundaries | 
| Phase 2 | Head + later backbone layers | Adapt high-level features | 
| Phase 3 | Full network | Global refinement |
---

## 🔬 Task 7: Progressive Training Experiment

### Procedure
1. Initialize with ImageNet weights
2. Train head-only
3. Gradually unfreeze layers
4. Retain learned weights across phases

📌 **Important:**  
Do NOT reinitialize the model between phases.

---

## 📊 Task 8: Comparative Analysis

Compare:
- Best separate strategy
- Progressive fine-tuning result

Evaluate:
- Final accuracy
- Total epochs
- Learning curve smoothness
- Generalization gap

---

In [ ]:
class ProgressiveTransferLearner:
    """
    Single model trained sequentially across 3 phases.
    Weights carry over between phases — this is NOT 3 separate models.
    """

    def __init__(self, num_classes=10, device='cuda'):
        self.device = device
        self.num_classes = num_classes
        self.history = {
            'phase1': None,
            'phase2': None,
            'phase3': None
        }
        self.best_accs = {}
        self._weight_snapshot = None  # for continuity verification

        self.model = self._build_model()
        self.criterion = nn.CrossEntropyLoss()

        print(f"  ✅ EfficientNet-B0 loaded, classifier replaced")
        print(f"  📊 Total parameters: {sum(p.numel() for p in self.model.parameters()):,}")

    def _build_model(self):
        model = models.efficientnet_b0(pretrained=True)
        in_features = model.classifier[-1].in_features
        model.classifier[-1] = nn.Linear(in_features, self.num_classes)
        return model.to(self.device)

    # ----------------------------------------------------------
    # Weight Continuity Verification
    # ----------------------------------------------------------

    def _snapshot_weights(self):
        """Save a snapshot of classifier weights before phase transition"""
        self._weight_snapshot = self.model.classifier[-1].weight.data.clone()

    def _verify_weight_continuity(self, phase_name):
        """
        Confirm weights carried over by comparing classifier weights
        to snapshot taken at end of previous phase.
        They must be identical at the START of the new phase
        before any gradient update.
        """
        if self._weight_snapshot is None:
            return

        current = self.model.classifier[-1].weight.data
        are_equal = torch.equal(current, self._weight_snapshot)

        print(f"\n  🔍 Weight Continuity Check — entering {phase_name}")
        print(f"     Classifier weights match previous phase end: "
              f"{'✅ YES — weights carried over correctly' if are_equal else '❌ NO — weights were reset'}")

        # Also show a numeric fingerprint for transparency
        print(f"     Weight tensor L2 norm: {current.norm().item():.6f} "
              f"(snapshot: {self._weight_snapshot.norm().item():.6f})")

    # ----------------------------------------------------------
    # Freeze / Unfreeze Utilities
    # ----------------------------------------------------------

    def _freeze_all(self):
        for param in self.model.parameters():
            param.requires_grad = False

    def _unfreeze_head(self):
        self._freeze_all()
        for param in self.model.classifier.parameters():
            param.requires_grad = True

    def _unfreeze_partial(self):
        self._freeze_all()
        for param in self.model.classifier.parameters():
            param.requires_grad = True
        for i in [5, 6, 7]:
            for param in self.model.features[i].parameters():
                param.requires_grad = True

    def _unfreeze_full(self):
        for param in self.model.parameters():
            param.requires_grad = True

    def _print_phase_stats(self, phase_name):
        total = sum(p.numel() for p in self.model.parameters())
        trainable = sum(p.numel() for p in self.model.parameters() if p.requires_grad)
        print(f"  📊 {phase_name}: {trainable:,} / {total:,} trainable ({trainable/total:.1%})")

    # ----------------------------------------------------------
    # Core Training Loop
    # ----------------------------------------------------------

    def _run_phase(self, phase_name, epochs, lr, train_loader, val_loader):
        print(f"\n{'='*50}")
        print(f"  {phase_name}")
        print(f"{'='*50}")
        self._print_phase_stats(phase_name)

        optimizer = optim.Adam(
            [p for p in self.model.parameters() if p.requires_grad],
            lr=lr
        )
        scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

        history = {
            'train_loss': [], 'train_acc': [],
            'val_loss':   [], 'val_acc':   [],
            'lr':         []
        }
        best_val_acc = 0.0

        for epoch in range(epochs):
            # --- Train ---
            self.model.train()
            train_loss, correct, total = 0.0, 0, 0

            pbar = tqdm(train_loader, desc=f'Epoch {epoch+1}/{epochs} [Train]')
            for images, labels in pbar:
                images, labels = images.to(self.device), labels.to(self.device)
                optimizer.zero_grad()
                outputs = self.model(images)
                loss = self.criterion(outputs, labels)
                loss.backward()
                optimizer.step()

                train_loss += loss.item()
                _, predicted = outputs.max(1)
                total += labels.size(0)
                correct += predicted.eq(labels).sum().item()
                pbar.set_postfix({
                    'Loss': f'{loss.item():.3f}',
                    'Acc':  f'{100.*correct/total:.1f}%'
                })

            avg_train_loss = train_loss / len(train_loader)
            train_acc = 100. * correct / total

            # --- Validate ---
            self.model.eval()
            val_loss, correct, total = 0.0, 0, 0

            with torch.no_grad():
                pbar = tqdm(val_loader, desc=f'Epoch {epoch+1}/{epochs} [Val]')
                for images, labels in pbar:
                    images, labels = images.to(self.device), labels.to(self.device)
                    outputs = self.model(images)
                    loss = self.criterion(outputs, labels)
                    val_loss += loss.item()
                    _, predicted = outputs.max(1)
                    total += labels.size(0)
                    correct += predicted.eq(labels).sum().item()
                    pbar.set_postfix({'Acc': f'{100.*correct/total:.1f}%'})

            avg_val_loss = val_loss / len(val_loader)
            val_acc = 100. * correct / total
            current_lr = optimizer.param_groups[0]['lr']
            scheduler.step()

            history['train_loss'].append(avg_train_loss)
            history['train_acc'].append(train_acc)
            history['val_loss'].append(avg_val_loss)
            history['val_acc'].append(val_acc)
            history['lr'].append(current_lr)

            print(f"\n  📈 Epoch {epoch+1} | "
                  f"Train Loss: {avg_train_loss:.4f} | Train Acc: {train_acc:.2f}% | "
                  f"Val Loss: {avg_val_loss:.4f} | Val Acc: {val_acc:.2f}% | "
                  f"LR: {current_lr:.6f}")

            if val_acc > best_val_acc:
                best_val_acc = val_acc
                torch.save(self.model.state_dict(),
                           f'best_{phase_name.lower().replace(" ", "_").replace("+","").replace("[","").replace("]","").replace(",","")}.pth')
                print(f"  🏆 New best! Val Acc: {best_val_acc:.2f}%")

        print(f"\n✅ {phase_name} complete | Best Val Acc: {best_val_acc:.2f}%")
        return history, best_val_acc

    # ----------------------------------------------------------
    # Progressive Phases
    # ----------------------------------------------------------

    def run(self, train_loader, val_loader):

        # ── Phase 1 ──────────────────────────────────────────
        self._unfreeze_head()
        h1, acc1 = self._run_phase(
            phase_name='Phase 1 — Head Only',
            epochs=5, lr=0.001,
            train_loader=train_loader, val_loader=val_loader
        )
        self.history['phase1'] = h1
        self.best_accs['phase1'] = acc1
        self._snapshot_weights()  # capture weights at end of phase 1

        # ── Phase 2 ──────────────────────────────────────────
        self._unfreeze_partial()
        self._verify_weight_continuity('Phase 2')  # verify before any update
        h2, acc2 = self._run_phase(
            phase_name='Phase 2 — Head + features[5,6,7]',
            epochs=5, lr=0.0005,
            train_loader=train_loader, val_loader=val_loader
        )
        self.history['phase2'] = h2
        self.best_accs['phase2'] = acc2
        self._snapshot_weights()  # capture weights at end of phase 2

        # ── Phase 3 ──────────────────────────────────────────
        self._unfreeze_full()
        self._verify_weight_continuity('Phase 3')  # verify before any update
        h3, acc3 = self._run_phase(
            phase_name='Phase 3 — Full Network',
            epochs=10, lr=0.0001,
            train_loader=train_loader, val_loader=val_loader
        )
        self.history['phase3'] = h3
        self.best_accs['phase3'] = acc3

        # ── Phase Summary ─────────────────────────────────────
        print("\n" + "="*60)
        print("PROGRESSIVE FINE-TUNING SUMMARY")
        print("="*60)
        print(f"  Phase 1 — Head Only          : {acc1:.2f}%")
        print(f"  Phase 2 — Partial Unfreeze   : {acc2:.2f}%")
        print(f"  Phase 3 — Full Fine-Tune     : {acc3:.2f}%")
        print(f"  Total Gain (P1 → P3)         : +{acc3 - acc1:.2f}%")
        print("="*60)

        return self.history

    # ----------------------------------------------------------
    # Task 8 — Comparative Analysis
    # ----------------------------------------------------------

    def compare_with_task6(self, task6_results: dict):
        """
        Task 8: Compare progressive fine-tuning against
        the three independent strategies from Task 6.

        Args:
            task6_results: trainer.results dict from Task 6
                           keys: 'head_only', 'partial', 'full'
                           each has: history, best_val_acc
        """
        print("\n" + "="*60)
        print("TASK 8 — COMPARATIVE ANALYSIS")
        print("="*60)

        # ── 1. Final Accuracy ─────────────────────────────────
        print("\n📊 1. FINAL ACCURACY COMPARISON")
        print("-"*50)

        task6_best_name = max(task6_results, key=lambda k: task6_results[k]['best_val_acc'])
        task6_best_acc  = task6_results[task6_best_name]['best_val_acc']
        progressive_acc = self.best_accs['phase3']

        print(f"  Task 6 — Head Only  : {task6_results['head_only']['best_val_acc']:.2f}%")
        print(f"  Task 6 — Partial    : {task6_results['partial']['best_val_acc']:.2f}%")
        print(f"  Task 6 — Full       : {task6_results['full']['best_val_acc']:.2f}%")
        print(f"  {'─'*40}")
        print(f"  Task 6 Best         : {task6_best_acc:.2f}% ({task6_best_name})")
        print(f"  Progressive (Ours)  : {progressive_acc:.2f}%")
        print(f"  Δ Accuracy          : {progressive_acc - task6_best_acc:+.2f}%")

        # ── 2. Total Epochs ───────────────────────────────────
        print("\n⏱️  2. TOTAL EPOCHS COMPARISON")
        print("-"*50)

        task6_epochs = {
            'head_only': len(task6_results['head_only']['history']['train_acc']),
            'partial':   len(task6_results['partial']['history']['train_acc']),
            'full':      len(task6_results['full']['history']['train_acc']),
        }
        progressive_total_epochs = (
            len(self.history['phase1']['train_acc']) +
            len(self.history['phase2']['train_acc']) +
            len(self.history['phase3']['train_acc'])
        )

        print(f"  Task 6 — Head Only  : {task6_epochs['head_only']} epochs")
        print(f"  Task 6 — Partial    : {task6_epochs['partial']} epochs")
        print(f"  Task 6 — Full       : {task6_epochs['full']} epochs")
        print(f"  Task 6 Total        : {sum(task6_epochs.values())} epochs (all 3 strategies combined)")
        print(f"  Progressive Total   : {progressive_total_epochs} epochs (single model)")

        # ── 3. Learning Curve Smoothness ──────────────────────
        print("\n📈 3. LEARNING CURVE SMOOTHNESS")
        print("-"*50)
        print("  (Measured as std of epoch-to-epoch val_acc changes — lower = smoother)")

        def smoothness(acc_list):
            diffs = [abs(acc_list[i+1] - acc_list[i]) for i in range(len(acc_list)-1)]
            return sum(diffs) / len(diffs) if diffs else 0.0

        for key in ['head_only', 'partial', 'full']:
            s = smoothness(task6_results[key]['history']['val_acc'])
            print(f"  Task 6 — {key:<10}: {s:.3f}%/epoch avg change")

        # Progressive — treat as one continuous curve
        progressive_combined = (
            self.history['phase1']['val_acc'] +
            self.history['phase2']['val_acc'] +
            self.history['phase3']['val_acc']
        )
        prog_smoothness = smoothness(progressive_combined)
        print(f"  Progressive         : {prog_smoothness:.3f}%/epoch avg change")

        # ── 4. Generalization Gap ─────────────────────────────
        print("\n🔍 4. GENERALIZATION GAP (Train Acc − Val Acc, final epoch)")
        print("-"*50)
        print("  (Closer to 0 = better generalization, negative = val > train)")

        for key in ['head_only', 'partial', 'full']:
            h = task6_results[key]['history']
            gap = h['train_acc'][-1] - h['val_acc'][-1]
            print(f"  Task 6 — {key:<10}: {gap:+.2f}%")

        prog_gap = (self.history['phase3']['train_acc'][-1] -
                    self.history['phase3']['val_acc'][-1])
        print(f"  Progressive         : {prog_gap:+.2f}%")

        # ── 5. Verdict ────────────────────────────────────────
        print("\n🏁 5. VERDICT")
        print("-"*50)
        winner = "Progressive" if progressive_acc > task6_best_acc else f"Task 6 ({task6_best_name})"
        print(f"  Best overall accuracy : {winner}")
        print(f"  Most efficient        : {'Progressive' if progressive_total_epochs < sum(task6_epochs.values()) else 'Task 6 (individual)'}")
        print(f"  Best generalization   : {'Progressive' if abs(prog_gap) < abs(task6_results[task6_best_name]['history']['train_acc'][-1] - task6_results[task6_best_name]['history']['val_acc'][-1]) else f'Task 6 ({task6_best_name})'}")
        print("="*60)


# ----------------------------------------------------------
# Run Progressive Learner
# ----------------------------------------------------------

progressive_learner = ProgressiveTransferLearner(num_classes=10, device=device)
history = progressive_learner.run(train_loader, val_loader)

# Task 8 — pass Task 6 results from the trainer object
progressive_learner.compare_with_task6(trainer.results)

## 📊 Task 8: Comparative Analysis

Compare:
- Best separate strategy
- Progressive fine-tuning result

Evaluate:
- Final accuracy
- Total epochs
- Learning curve smoothness
- Generalization gap

---

## 🧠 Interpretation

### Why Progressive Fine-Tuning Works

1. **Weight continuity** preserves task-relevant features  
2. **Gradual adaptation** minimizes representation shock  
3. **Efficient optimization** avoids redundant relearning  
4. **Improved feature–classifier alignment** enhances generalization  

---

## 📘 Task 9: Hands-On Machine Learning (Aurélien Géron)

### Chapter 12 — Neural Networks with PyTorch

#### Your Tasks
- Implement all **end-of-chapter exercises**
- Add:
  - Code
  - Explanations
  - Observations

📌 Integrate solutions directly into this notebook.

### 1. What are the advantages of a CNN over a fully connected DNN for image classification?
    By flattening an image immediately in a standard DNN, we destroy all the img spatial information—the physical relationships between neighboring pixels. CNNs solve this by using kernels to scan the image in its dimensional state, extracting meaningful features (like edges, textures, and eventually complex shapes) before flattening those highly distilled feature maps.

### 2. Consider a CNN composed of three convolutional layers,each with 3 × 3 kernels, a stride of 2, and "same" padding.The lowest layer outputs 100 feature maps, the middle oneoutputs 200, and the top one outputs 400. The inputimages are RGB images of 200 × 300 pixels:

### a. What is the total number of parameters in theCNN?
### b. If we are using 32-bit floats, at least how much RAM will this network require when making a prediction for a single instance?
### c. What about when training on a mini-batch of 50 images?






### Layer outputs:

* Input: **200 × 300**
* Layer 1 → **100 × 150 × 100**
* Layer 2 → **50 × 75 × 200**
* Layer 3 → **25 × 38 × 400**
  (75 / 2 = 37.5 → ceil = 38)

---

### ✅ (a) Total Number of Parameters


Parameters=((Kernel Width×Kernel Height×Input Channels)+1 Bias)×Output Channels
Let's apply this to our three layers (using the 3×3 kernels):

Layer 1:
((3×3×3)+1)×100=(27+1)×100= 2,800 parameters

Layer 2: 
((3×3×100)+1)×200=(900+1)×200= 180,200 parameters

Layer 3: 

((3×3×200)+1)×400=(1800+1)×400= 720,400 parameters

    Total: 2,800 + 180,200 + 720,400 = 903,400 total parameters.



### ✅ (b) RAM for One Prediction

We must store:
* parameters
* activations (feature maps)
---
## 🔸 Parameters memory
Each parameter = **32 bits = 4 bytes**
903,400 x 4 = 3,613,600 
---
## 🔸 Activations
### Input:
200 x 300 x 3 = 180,000
### Layer 1:
100 x 150 x 100 = 1,500,000
### Layer 2:
50 x 75 x 200 = 750,000
### Layer 3:
25 x 38 x 400 = 380,000
---

### Total activations:

180,000 + 1,500,000 + 750,000 + 380,000 = 2,810,000
Memory:
    2,810,000 x 4 = 11,240,000bytes  11.24MB
---
## 🔥 Total RAM (inference):

### 3.6 + 11.24 = ~14.8 MB

---

# ✅ (c) Training with Batch Size = 50

During training:

* Need activations for **all batch**
* Plus gradients (≈ same size as activations)

---

## 🔸 Activations:

11.24 x 50 = 562 MB


---

## 🔸 Gradients:

≈ same → **+562 MB**

---

## 🔸 Parameters:

~3.6 MB (negligible compared to above)

---

## 🔥 Total:

562 + 562 + 3.6 = ~1127MB  = 1.1 GB



### 3. If your GPU runs out of memory while training a CNN, what are five things you could try to solve the problem?
If we training process crashes due to an out-of-memory error and you haven't saved your progress, we will unfortunately have to start over 

    there are several strategies you can employ to reduce your model's memory footprint:
    Reduce the batch size and accumulate gradients
    Reduce dimensionality or remove layers
    Use 16-bit floats instead of 32-bit floats
    Distribute the CNN or offload to the CPU
    Trade compute for lower memory usage



### 4. Why would you want to add a max pooling layer rather than a convolutional layer with the same stride?
    Zero Parameters
    Max pooling is an intentionally destructive process that preserves only the strongest features in a given receptive field while getting rid of all the weaker, meaningless signals
    This ensures that the subsequent layers are fed a much cleaner, more prominent feature map to work with

### 5. Can you name the main innovations in AlexNet, as compared to LeNet-5? What about the main innovations in GoogLeNet, ResNet, SENet, Xception, EfficientNet, and ConvNeXt?

    🔹 AlexNet (vs LeNet-5)
    Deeper & larger network
    Stacked conv layers (not always conv → pool)
    ReLU, Dropout, Data augmentation
    LRN (local competition)
    🔹 GoogLeNet (Inception)
    Inception modules → parallel filters (1×1, 3×3, 5×5)
    1×1 bottlenecks → reduce computation
    Global average pooling (no big FC layers)
    🔹 ResNet
    Skip connections
    👉 enables very deep networks (fixes vanishing gradients)
    🔹 SENet
    Channel attention
    👉 learns which feature maps are important
    🔹 Xception
    Depthwise separable convolutions
    👉 separates spatial & channel processing (more efficient)
    🔹 EfficientNet
    Compound scaling
    👉 scales depth + width + resolution together
    🔹 ConvNeXt
     Updated the ResNet architecture by drawing direct inspiration from vision transformer architectures




### 6. What is a fully convolutional network? How can you convert a dense layer into a convolutional layer?
     🔹 What is an FCN?
    A CNN with no dense (fully connected) layers
    Uses only convolution + pooling
    🔹 Main Advantage
    Works with any image size
    Outputs a grid of predictions (not just one output)
    Much more efficient than sliding a CNN over an image
    🔹 Key Idea

    👉 A normal CNN = makes one prediction
    👉 FCN = makes many predictions across the image at once

    To convert a dense layer into a convolutional layer, you must configure the new convolutional layer with specific hyperparameters:
    The number of filters must be equal to the number of units in the original dense layer
    The filter size must perfectly match the spatial size (height and width) of the input feature maps
    The padding must be set to "valid"
    The stride may be set to 1 or more

### 7. What is the main technical difficulty of semantic segmentation?

    Semantic segmentation is a computer vision task where every single pixel in an image is classified according to the type of object it belongs to (such as a road, car, or building). This is different from image classification, which assigns a single label to an entire image, and object detection, which identifies and localizes objects with bounding boxes.
    The main technical difficulty of semantic segmentation is that images gradually lose their spatial resolution as they are processed through a regular Convolutional Neural Network (CNN)
    . This loss of fine-grained spatial detail is caused by layers within the network that use strides greater than 1




### 8. Build your own CNN from scratch and try to achieve thehighest possible accuracy on MNIST.

In [ ]:


device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"  Device: {device}")

# ----------------------------------------------------------
# 1. Data
# ----------------------------------------------------------

train_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
    # No augmentation for linear — it doesn't benefit from it
])

test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

train_dataset = torchvision.datasets.MNIST(root='./data', train=True,
                                            download=True, transform=train_transform)
test_dataset  = torchvision.datasets.MNIST(root='./data', train=False,
                                            download=True, transform=test_transform)

train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True,
                          num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=256, shuffle=False,
                          num_workers=2, pin_memory=True)

print(f"  Train: {len(train_dataset):,} | Test: {len(test_dataset):,}")

# ----------------------------------------------------------
# 2. Model
# ----------------------------------------------------------
class BaselineLinear(nn.Module):
    def __init__(self):
        super().__init__()

        # ── One Conv Block ────────────────────────────────────
        self.conv_block = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),  # 28×28 → 28×28
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2)                            # 28×28 → 14×14
        )

        # ── Linear Head ───────────────────────────────────────
        self.head = nn.Sequential(
            nn.Flatten(),                  # 32 × 14 × 14 = 6272
            nn.Linear(32 * 14 * 14, 256),
            nn.ReLU(),
            nn.Linear(256, 10)
        )

    def forward(self, x):
        x = self.conv_block(x)
        x = self.head(x)
        return x

# ----------------------------------------------------------
# 3. Trainer
# ----------------------------------------------------------

class MnistTrainer:

    def __init__(self, model, model_name, device):
        self.model      = model.to(device)
        self.model_name = model_name
        self.device     = device
        self.history    = {'train_loss': [], 'train_acc': [],
                           'val_loss':   [], 'val_acc':   []}
        self.best_acc   = 0.0
        self.criterion  = nn.CrossEntropyLoss()

    def train(self, train_loader, val_loader, epochs=10, lr=0.001):
        optimizer = optim.Adam(self.model.parameters(), lr=lr, weight_decay=1e-4)
        scheduler = optim.lr_scheduler.OneCycleLR(
            optimizer,
            max_lr=lr,
            steps_per_epoch=len(train_loader),
            epochs=epochs,
            pct_start=0.3
        )

        print(f"\n🚀 Training {self.model_name}")
        print(f"   Params: {sum(p.numel() for p in self.model.parameters()):,}")
        print("="*50)

        for epoch in range(epochs):
            # --- Train ---
            self.model.train()
            train_loss, correct, total = 0.0, 0, 0

            pbar = tqdm(train_loader, desc=f'Epoch {epoch+1}/{epochs} [Train]')
            for images, labels in pbar:
                images, labels = images.to(self.device), labels.to(self.device)

                optimizer.zero_grad()
                outputs = self.model(images)
                loss = self.criterion(outputs, labels)
                loss.backward()
                nn.utils.clip_grad_norm_(self.model.parameters(), max_norm=1.0)
                optimizer.step()
                scheduler.step()

                train_loss += loss.item()
                _, predicted = outputs.max(1)
                total   += labels.size(0)
                correct += predicted.eq(labels).sum().item()
                pbar.set_postfix({
                    'Loss': f'{loss.item():.3f}',
                    'Acc':  f'{100.*correct/total:.2f}%'
                })

            avg_train_loss = train_loss / len(train_loader)
            train_acc      = 100. * correct / total

            # --- Validate ---
            val_loss, val_acc = self._evaluate(val_loader)

            self.history['train_loss'].append(avg_train_loss)
            self.history['train_acc'].append(train_acc)
            self.history['val_loss'].append(val_loss)
            self.history['val_acc'].append(val_acc)

            print(f"\n  📈 Epoch {epoch+1:>2} | "
                  f"Train Loss: {avg_train_loss:.4f} | Train Acc: {train_acc:.2f}% | "
                  f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.2f}% | "
                  f"LR: {scheduler.get_last_lr()[0]:.6f}")

            if val_acc > self.best_acc:
                self.best_acc = val_acc
                torch.save(self.model.state_dict(), f'best_{self.model_name}.pth')
                print(f"  🏆 New best! {self.best_acc:.2f}% — saved")

        print(f"\n✅ Done | Best Val Acc: {self.best_acc:.2f}%")

    def _evaluate(self, loader):
        self.model.eval()
        total_loss, correct, total = 0.0, 0, 0
        with torch.no_grad():
            for images, labels in loader:
                images, labels = images.to(self.device), labels.to(self.device)
                outputs = self.model(images)
                loss    = self.criterion(outputs, labels)
                total_loss += loss.item()
                _, predicted = outputs.max(1)
                total   += labels.size(0)
                correct += predicted.eq(labels).sum().item()
        return total_loss / len(loader), 100. * correct / total

    def get_predictions(self, loader):
        self.model.eval()
        all_preds, all_labels = [], []
        with torch.no_grad():
            for images, labels in loader:
                images = images.to(self.device)
                outputs = self.model(images)
                _, predicted = outputs.max(1)
                all_preds.extend(predicted.cpu().numpy())
                all_labels.extend(labels.numpy())
        return np.array(all_preds), np.array(all_labels)

# ----------------------------------------------------------
# 4. Analysis
# ----------------------------------------------------------

def plot_confusion_matrix(preds, labels, model_name):
    cm = confusion_matrix(labels, preds)
    fig, ax = plt.subplots(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=range(10), yticklabels=range(10), ax=ax)
    ax.set_xlabel('Predicted')
    ax.set_ylabel('True')
    ax.set_title(f'Confusion Matrix — {model_name}')
    plt.tight_layout()
    plt.savefig(f'confusion_matrix_{model_name}.png', dpi=150)
    plt.show()

def plot_learning_curves(history, model_name):
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    axes[0].plot(history['train_loss'], label='Train', color='steelblue')
    axes[0].plot(history['val_loss'],   label='Val',   color='orange', linestyle='--')
    axes[0].set_title(f'Loss — {model_name}')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Loss')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

    axes[1].plot(history['train_acc'], label='Train', color='steelblue')
    axes[1].plot(history['val_acc'],   label='Val',   color='orange', linestyle='--')
    axes[1].set_title(f'Accuracy — {model_name}')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Accuracy (%)')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(f'learning_curves_{model_name}.png', dpi=150)
    plt.show()

def print_summary(trainer):
    preds, labels = trainer.get_predictions(test_loader)

    print(f"\n📋 Per-Class Report — {trainer.model_name}")
    print("-"*50)
    print(classification_report(labels, preds,
                                 target_names=[str(i) for i in range(10)]))

    gen_gap = trainer.history['train_acc'][-1] - trainer.history['val_acc'][-1]
    print(f"  Best Val Acc      : {trainer.best_acc:.2f}%")
    print(f"  Generalization Gap: {gen_gap:+.2f}%")
    print(f"  Total Params      : {sum(p.numel() for p in trainer.model.parameters()):,}")

    plot_confusion_matrix(preds, labels, trainer.model_name)
    plot_learning_curves(trainer.history, trainer.model_name)

# ----------------------------------------------------------
# 5. Run
# ----------------------------------------------------------

trainer = MnistTrainer(BaselineLinear(), 'Baseline_Linear', device)
trainer.train(train_loader, test_loader, epochs=10, lr=0.001)
print_summary(trainer)

# PART 6 — Final Discussion

## 📌 When to Use Each Approach

| Scenario | Recommended Strategy | Reason |
|--------|---------------------|--------|
| Production systems | Progressive fine-tuning | Best accuracy & stability |
| Limited compute | Progressive fine-tuning | Efficient use of epochs |
| Academic ablations | Separate training | Clean isolation |
| Rapid baselines | Head-only | Fastest |

---

---

## 🎯 Final Conclusion

Progressive transfer learning provides a **realistic, efficient, and professional training paradigm** by **building on learned knowledge instead of discarding it**.

This notebook demonstrates that:

> **Gradual adaptation consistently outperforms isolated fine-tuning strategies**, especially in real-world scenarios.

---

## 🧠 Follow-up Research Questions

- # Does progressive fine-tuning always outperform separate training?
            o, progressive fine-tuning does not always outperform separate training. Its effectiveness depends on factors such as dataset size, task similarity, and risk of overfitting. In some cases, simpler approaches or full retraining may perform equally well or better.

- # What is the optimal epoch allocation per phase?
            There is no fixed optimal epoch allocation. Typically, a few epochs are used for training the top layers, followed by more epochs as deeper layers are progressively unfrozen.

- # How does dataset size affect progressive fine-tuning benefits?7
            With small datasets, progressive fine-tuning is beneficial because it reduces overfitting and preserves pretrained features. With large datasets, its advantage decreases, as the model can be fully fine-tuned effectively without needing gradual unfreezing.

- # How early should deeper layers be unfrozen?
            Deeper layers should be unfrozen only after the top layers have stabilized, typically after a few epochs. Unfreezing too early can destabilize training and destroy pretrained features, so layers are usually unfrozen gradually with a lower learning rate
            👉 Early layers = general features (edges, textures)
            👉 Late layers = task-specific

---

## ✅ Deliverables Checklist

- [ ] CIFAR-10 pipeline (from scratch)
- [ ] Scratch CNN experiments
- [ ] Transfer learning (3 strategies)
- [ ] Progressive fine-tuning experiment
- [ ] Géron Chapter 12 exercises
- [ ] Comparative analysis & conclusions

---